# 🤖 – Soft Robotics Cybersecurity Lab
## Micro-Perimeters, SROS 2 Governance, and Cyber-Physical ADS

- **Format:** Individual or Pairs
- **Evaluation:** Live CTF Challenge (Min 65) + Gamified Quiz (Min 75)

In [1]:
# BLOCK 0: Environment & Test Harness Setup
# =====================================================================
# Run this cell to prepare dependencies and load evaluation structures.
# =====================================================================
import sys
import time

print("âœ… Environment setup complete.")

âœ… Environment setup complete.


--- 
## ðŸ§© BLOCK 1: IEC 62443 Architecture & Layer 3 vs. Layer 7 Security (15 min)

### ðŸ“ TASK 1.1: Threat Modeling & Protocol Analysis
Read the industrial scenario and answer the following questions:

1. **Defense-in-Depth Analysis:** An attacker compromises an Engineering Workstation on **VLAN 50** and attempts an IP spoofing attack to inject DDS packets into the soft robot's local bus (**VLAN 10**, subnet `192.168.20.0/24`). 

Explain why an L3/L4 perimeter firewall configured with `Default-Deny` stops cross-VLAN traffic, but fails to protect the robot if the attacker connects directly to an unauthenticated Wi-Fi access point bridged inside VLAN 10.

> **Write your answer here:**
> 

2. **DDS-Security Mechanics:** How does SROS 2 prevent topic hijacking at Layer 7 even when an attacker is physically attached to VLAN 10 and sending valid UDP packets? At what cryptographic handshake phase is unauthorized traffic dropped?

> **Write your answer here:**
> 

# ðŸ§© BLOCK 2: Host Firewall Rule Audit & Remediation (20 min)

### TASK 2: AUDIT & CORRECTION
The class below emulates a host firewall filter. 
It contains 2 CRITICAL SECURITY FLAWS

1. Identify the flaws in the text cell below.
2. Complete 'CorrectedNftablesEvaluator' to fix all vulnerabilities.

In [3]:
class DefectiveNftablesEvaluator:
    def __init__(self):
        self.allowed_dds_subnet = "192.168.20."

    def evaluate_packet(self, src_ip: str, dport: int, uid: int, is_raw_socket: bool):
        access = "DROP"
        if (7400 <= dport <= 7500) and src_ip.startswith(self.allowed_dds_subnet):
            access = "ACCEPT"
        
        if uid != 0 and is_raw_socket:
            return "DROP"
        elif uid == 0 and is_raw_socket:
            access = "ACCEPT" 
            
        return access

# ---- RUN AUDIT TESTS ----
evaluator = DefectiveNftablesEvaluator()
print("Test A (DDS Port 7405):", evaluator.evaluate_packet("192.168.20.10", 7405, uid=1000, is_raw_socket=False))
print("Test B (Raw Socket UID 1000):", evaluator.evaluate_packet("10.0.0.5", 80, uid=1000, is_raw_socket=True))

Test A (DDS Port 7405): ACCEPT
Test B (Raw Socket UID 1000): DROP


### ðŸ“ AUDIT FINDINGS
Identify the two flaws in `DefectiveNftablesEvaluator` and explain the security risk of Flaw 2:

> **Flaw 1:** 
> 
> **Flaw 2:** 

In [ ]:
# =====================================================================
# âœï¸ STUDENT CODE REQUIRED: Advanced Micro-Perimeter Rule Engine
#
# IMPLEMENT 'audit_telemetry_flow':
# 1. HARDENED RAW SOCKET RULE: Reject ANY raw socket packet if process UID != 0,
#    REGARDLESS of the interface, IP address, or port requested.
#
# 2. STRICT INTERFACE & SUBNET VALIDATION:
#    - DDS Discovery traffic MUST ONLY be accepted from interface "eth0".
#    - Transport protocol MUST be "UDP".
#    - Source IP must be strictly inside the 192.168.20.0/24 subnet.
#    - Destination port must fall strictly within the DDS Discovery range.
#    - Loopback Isolation: Allow 'lo' interface traffic, block external spoofing.
#
# 3. DEFAULT-DENY: Any packet not explicitly whitelisted must return "DROP".

# =====================================================================

class CorrectedNftablesEvaluator:
    def __init__(self):
        # TODO: Define subnet parameters and valid port ranges
        pass

    def audit_telemetry_flow(self, net_origin: str, svc_channel: int, proc_auth_id: int, 
                             is_raw_socket: bool, egress_iface: str, transport_proto: str) -> str:
        # TODO: Implement strict multi-variable network filtering
        pass

# check

## ðŸ§© BLOCK 3: SROS 2 Policy Audit & Wildcard Exploitation (25 min)

### TASK 3: Fix the permission file below.


In [ ]:

%%writefile defective_permissions.xml
<?xml version="1.0" encoding="UTF-8"?>
<dds xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <permissions>
    <grant name="soft_gripper_grant">
      <subject_name>CN=/soft_gripper_monitoring_node</subject_name>
      <validity><not_before>2026-01-01</not_before><not_after>2028-01-01</not_after></validity>
      
      <allow>
        <publish>
          <topics>
            <topic>/soft_robot/*</topic>
          </topics>
        </publish>
      </allow>
      <default>DENY</default>
    </grant>
  </permissions>
</dds>

### ðŸ“ TASK 3.1: Wildcard Vulnerability Analysis
Explain why using `<topic>/soft_robot/*</topic>` violates the Principle of Least Privilege when the node only needs to publish to `/soft_robot/telemetry/pressure`. Which sensitive command topics are accidentally exposed?

> **Write your answer here:**
> 

In [ ]:
%%writefile permissions.xml
<!-- âœï¸ STUDENT XML REQUIRED: Write the hardened permissions.xml file -->
<!-- 1. Allow publishing ONLY to /soft_robot/telemetry/pressure -->
<!-- 2. Allow subscribing ONLY to /parameter_events -->
<!-- 3. Explicitly/implicitly DENY publishing to /soft_robot/pressure_cmd -->
<!-- 4. Set default policy to DENY -->

<dds xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <permissions>
    <grant name="soft_gripper_grant">
      <!-- TODO: Fill in grant details and restrictive allow blocks -->
    </grant>
  </permissions>
</dds>

## ðŸ§© BLOCK 4: Pneumatic Anomaly Detection System (ADS) (20 min)


### TASK 4: CYBER-PHYSICAL MONITORING FOR SOFT ROBOTICS

Based on the facility specifications:
1. Calculate the maximum safe operating pressure (MAX_PRESSURE_BAR) 
   given a burst threshold of 2.5 bar and a 20% safety margin.
2. Set the maximum valve switching frequency threshold .
3. Program the static overpressure and valve rate anomaly detection checks.


In [ ]:


import time

class SoftRobotPressureADS:
    def __init__(self):
        # âš ï¸ Burst pressure = 2.5 bar
        # TODO 1: Calculate maximum safe pressure with a 20% safety margin


        # TODO 2: Define maximum valve actuation frequency (Hz) based on scenario spec


        self.last_cmd_time = time.time()
        self.cmd_count = 0

    def inspect_pressure_command(self, current_pressure_bar: float) -> str:
        current_time = time.time()

        # -------------------------------------------------------------
        # TODO 3: Implement static overpressure detection check
        # -------------------------------------------------------------
        if :
            print(f"ðŸš¨ [SOFT-ROBOT ADS ALERT] OVERPRESSURE DETECTED IN FLUIDIC ACTUATOR!")
            print(f"   Requested: {current_pressure_bar:.2f} bar | Safe Limit: {self.MAX_PRESSURE_BAR:.2f} bar")
            print("ðŸ›‘ >>> TRIGGERING EMERGENCY DEPRESSURIZATION VALVE (E-RELEASE) <<<")
            return "E_RELEASE_STATIC_OVERPRESSURE"

        # -------------------------------------------------------------
        # TODO 4: Implement valve DoS / rapid cycling detection
        # -------------------------------------------------------------
        self.cmd_count += 1
        elapsed = current_time - self.last_cmd_time
        if elapsed >= 1.0:
            freq = self.cmd_count / elapsed
            if :
                print(f"âš ï¸ [SOFT-ROBOT ADS ALERT] HIGH-FREQUENCY VALVE CYCLING DETECTED!")
                print(f"   Command Rate: {freq:.1f} Hz | Fatigue Threshold: {self.MAX_VALVE_FREQ_HZ} Hz")
                return "VALVE_FATIGUE_ATTACK"
            self.cmd_count = 0
            self.last_cmd_time = current_time

        return "OK"

# ---- EVALUATION TEST RIG ----
# ads_soft = SoftRobotPressureADS()
# ads_soft.inspect_pressure_command(1.5) # Normal
# ads_soft.inspect_pressure_command(2.8) # Overpressure Alert